In [ ]:
import json
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime, timezone

BASE = Path("..")

FILES = {
    "usernames": BASE / "complete_dataset_raw/raw_usernames_compressed.jsonl",
    "phone_numbers": BASE / "complete_dataset_raw/raw_numbers_compressed.jsonl"
}

OUT = BASE / "cleaning/auction_check"
OUT.mkdir(parents=True, exist_ok=True)
# Cerca trasferimenti entro 24 ore dalla conclusione dell'asta.
# La stessa trace viene considerata anche fuori dalla finestra.
WINDOW_SECONDS = 24 * 3600

def norm_addr(value):
    return str(value).strip().lower() if value else None

def get_details(record):
    details = record.get("det") or {}
    return details if isinstance(details, dict) else {}

def readable_time(ts):
    return (
        datetime.fromtimestamp(ts, tz=timezone.utc).isoformat()
        if ts is not None else None
    )

In [2]:
def trova_aste_concluse(path):
    aste = defaultdict(list)
    stats = Counter()

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            r = json.loads(line)

            if r.get("y") != "nft_finish_auction":
                continue

            if r.get("ok") is not True:
                continue

            nft = norm_addr(r.get("n"))
            if not nft:
                continue

            det = get_details(r)

            asta = {
                "nft": nft,
                "finish_timestamp": r.get("t"),
                "finish_trace": r.get("tid"),
                "finish_action": r.get("aid"),
                "auction_address": norm_addr(det.get("auction_address")),
                "finish_owner": norm_addr(det.get("owner")),
                "best_match": None,
                "best_score": None,
                "candidate_count": 0,
                "bid_count": 0,
                "last_bidder": None,
                "last_bid_amount_ton": None,
                "last_bid_timestamp": None
            }

            aste[nft].append(asta)
            stats["finished"] += 1

    return aste, stats

In [3]:
def confronta_trasferimenti(path, aste, stats):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            r = json.loads(line)

            if r.get("y") not in {"nft_transfer", "nft_purchase"}:
                continue

            if r.get("ok") is not True:
                continue

            nft = norm_addr(r.get("n"))

            if nft not in aste:
                continue

            ts = r.get("t")
            trace = r.get("tid")
            det = get_details(r)

            for asta in aste[nft]:
                finish_ts = asta["finish_timestamp"]
                same_trace = bool(
                    trace and asta["finish_trace"]
                    and trace == asta["finish_trace"]
                )

                delta = (
                    abs(ts - finish_ts)
                    if isinstance(ts, int) and isinstance(finish_ts, int)
                    else None
                )

                if not same_trace and (
                    delta is None or delta > WINDOW_SECONDS
                ):
                    continue

                asta["candidate_count"] += 1

                # Priorità alla stessa trace; poi vicinanza temporale
                score = (
                    0 if same_trace else 1,
                    delta if delta is not None else float("inf"),
                    0 if r.get("y") == "nft_purchase" else 1
                )

                if asta["best_score"] is None or score < asta["best_score"]:
                    asta["best_score"] = score
                    asta["best_match"] = {
                        "matched_type": r.get("y"),
                        "matched_trace": trace,
                        "matched_action": r.get("aid"),
                        "matched_timestamp": ts,
                        "same_trace": same_trace,
                        "delta_seconds": delta,
                        "new_owner": norm_addr(det.get("new_owner")),
                        "old_owner": norm_addr(det.get("old_owner")),
                        "real_old_owner": norm_addr(det.get("real_old_owner")),
                        "price_ton": (
                            int(det["price"]) / 1e9
                            if det.get("price") is not None else None
                        )
                    }

In [4]:
def confronta_bid(path, aste, stats):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            r = json.loads(line)

            if r.get("y") != "auction_bid" or r.get("ok") is not True:
                continue

            nft = norm_addr(r.get("n"))
            if nft not in aste:
                continue

            det = get_details(r)
            ts = r.get("t")

            if not isinstance(ts, int):
                continue

            bid_auction = norm_addr(det.get("auction"))

            for asta in aste[nft]:
                finish_ts = asta["finish_timestamp"]

                if not isinstance(finish_ts, int) or ts > finish_ts:
                    continue

                # Se entrambi gli indirizzi d'asta sono disponibili,
                # devono coincidere.
                if (
                    bid_auction and asta["auction_address"]
                    and bid_auction != asta["auction_address"]
                ):
                    continue

                asta["bid_count"] += 1

                last_ts = asta["last_bid_timestamp"]

                if last_ts is None or ts > last_ts:
                    asta["last_bid_timestamp"] = ts
                    asta["last_bidder"] = norm_addr(det.get("bidder"))
                    asta["last_bid_amount_ton"] = (
                        int(det["amount"]) / 1e9
                        if det.get("amount") is not None else None
                    )

In [5]:
all_results = []

for layer, path in FILES.items():
    print(f"\nAnalizzo {layer}...")

    aste, stats = trova_aste_concluse(path)

    confronta_trasferimenti(path, aste, stats)
    confronta_bid(path, aste, stats)

    rows = []

    for nft, nft_aste in aste.items():
        for asta in nft_aste:
            match = asta["best_match"] or {}

            if match.get("same_trace"):
                status = "SAME_TRACE"
            elif match:
                status = "NEAR_TIMESTAMP"
            else:
                status = "NO_MATCH"

            bidder = asta["last_bidder"]
            recipient = match.get("new_owner")

            rows.append({
                "layer": layer,
                "nft": nft,
                "auction_address": asta["auction_address"],
                "finish_time": readable_time(asta["finish_timestamp"]),
                "status": status,
                "matched_type": match.get("matched_type"),
                "matched_time": readable_time(match.get("matched_timestamp")),
                "delta_seconds": match.get("delta_seconds"),
                "candidate_count": asta["candidate_count"],
                "bid_count": asta["bid_count"],
                "last_bidder": bidder,
                "recipient": recipient,
                "bidder_matches_recipient": (
                    bidder == recipient
                    if bidder and recipient else None
                ),
                "last_bid_amount_ton": asta["last_bid_amount_ton"],
                "transfer_price_ton": match.get("price_ton"),
                "finish_trace": asta["finish_trace"],
                "matched_trace": match.get("matched_trace"),
                "matched_action": match.get("matched_action")
            })

    df = pd.DataFrame(rows)
    df.to_csv(OUT / f"auction_matches_{layer}.csv", index=False)

    print(f"\nAste concluse: {len(df):,}")

    if not df.empty:
        display(
            df["status"]
            .value_counts()
            .rename_axis("status")
            .reset_index(name="n_aste")
        )

        print("\nCoincidenza ultimo bidder / destinatario:")
        display(df["bidder_matches_recipient"].value_counts(dropna=False))

    all_results.extend(rows)

df_all = pd.DataFrame(all_results)

print("\nControllo terminato.")
print("File salvati in:", OUT.resolve())


Analizzo usernames...



Aste concluse: 21,936


,status,n_aste
0,SAME_TRACE,21934
1,NEAR_TIMESTAMP,2



Coincidenza ultimo bidder / destinatario:


bidder_matches_recipient
None     17373
True      4537
False       26
Name: count, dtype: int64


Analizzo phone_numbers...



Aste concluse: 2,410


,status,n_aste
0,SAME_TRACE,2410



Coincidenza ultimo bidder / destinatario:


bidder_matches_recipient
None     1739
True      666
False       5
Name: count, dtype: int64


Controllo terminato.
File salvati in: /home/local/ADUNIPI/a.belliani/CNA/cleaning/auction_check
